In [1]:
import pandas as pd
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
import plotly.figure_factory as ff
import matplotlib.pyplot as plt

file_path = 'sleep.csv'
df = pd.read_csv(file_path)

display(df.head())

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [2]:
# Get the number of rows and columns so i can know what i am working with
num_rows, num_columns = df.shape

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")

Number of rows: 374
Number of columns: 13


In [3]:
(df.isnull().sum())

Person ID                    0
Gender                       0
Age                          0
Occupation                   0
Sleep Duration               0
Quality of Sleep             0
Physical Activity Level      0
Stress Level                 0
BMI Category                 0
Blood Pressure               0
Heart Rate                   0
Daily Steps                  0
Sleep Disorder             219
dtype: int64

In [4]:
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")


Number of duplicate rows: 0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Person ID                374 non-null    int64  
 1   Gender                   374 non-null    object 
 2   Age                      374 non-null    int64  
 3   Occupation               374 non-null    object 
 4   Sleep Duration           374 non-null    float64
 5   Quality of Sleep         374 non-null    int64  
 6   Physical Activity Level  374 non-null    int64  
 7   Stress Level             374 non-null    int64  
 8   BMI Category             374 non-null    object 
 9   Blood Pressure           374 non-null    object 
 10  Heart Rate               374 non-null    int64  
 11  Daily Steps              374 non-null    int64  
 12  Sleep Disorder           155 non-null    object 
dtypes: float64(1), int64(7), object(5)
memory usage: 38.1+ KB


In [3]:
# This shows a summary of the number-based data (like average, minimum, maximum, and the likes)
round (df.describe(exclude = 'object'), 2).style.background_gradient(cmap='BuPu')

,Person ID,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps
count,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000
mean,187.500000,42.180000,7.130000,7.310000,59.170000,5.390000,70.170000,6816.840000
std,108.110000,8.670000,0.800000,1.200000,20.830000,1.770000,4.140000,1617.920000
min,1.000000,27.000000,5.800000,4.000000,30.000000,3.000000,65.000000,3000.000000
25%,94.250000,35.250000,6.400000,6.000000,45.000000,4.000000,68.000000,5600.000000
50%,187.500000,43.000000,7.200000,7.000000,60.000000,5.000000,70.000000,7000.000000
75%,280.750000,50.000000,7.800000,8.000000,75.000000,7.000000,72.000000,8000.000000
max,374.000000,59.000000,8.500000,9.000000,90.000000,8.000000,86.000000,10000.000000


In [4]:
# This shows a summary of the text-based (categorical) data, like how many categories there are and which one appears most often.
round (df.describe(exclude = ['float', 'int64']),2).style.set_properties(**{'background-color': '#4A235A','color': '#E2EEF3'})  

,Gender,Occupation,BMI Category,Blood Pressure,Sleep Disorder
count,374,374,374,374,155
unique,2,11,4,25,2
top,Male,Nurse,Normal,130/85,Sleep Apnea
freq,189,73,195,99,78


In [11]:
sd = df['Sleep Disorder'].value_counts(normalize=True).mul(100).rename_axis('Sleep Disorder').reset_index(name='Percentage')
colors = dict(zip(sd['Sleep Disorder'], ['#636EFA', '#EF553B']))

fig = px.pie(sd, names='Sleep Disorder', values='Percentage', color='Sleep Disorder', color_discrete_map=colors, hole=0.4,
             title='Percentage of People With and Without Sleep Disorder')
fig.update_traces(textinfo='percent+label', pull=[0, 0.1], marker_line_color='black', marker_line_width=[0, 3])

fig.show()


In [ ]:
# The particular code helps efine stress categories with conditions and choices
conditions = [
    df['Stress Level'] == 3,
    df['Stress Level'].between(4, 5),
    df['Stress Level'].between(6, 7),
    df['Stress Level'] == 8
]
choices = ['Low Stress', 'Medium Stress', 'High Stress', 'Very High Stress']
df['Stress Category'] = np.select(conditions, choices, default='Unknown')

stress_colors = {'Low Stress': '#512b58', 'Medium Stress': '#fa8c3b', 'High Stress': '#fe346e', 'Very High Stress': '#b33b3b'}

bins = [0, 12, 18, 35, 50, 100]
labels = ['Children', 'Teens', 'Adults', 'Mid Adults', 'Elderly']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

fig1 = px.scatter(df, x='Age', y='Stress Level', color='Stress Category', hover_name='Age Group',
                  color_discrete_map=stress_colors, title="Impact of Age on Stress Level",
                  labels={"Age": "Age", "Stress Level": "Stress Level"}, size_max=10)
fig1.update_traces(marker=dict(size=10, line=dict(width=2, color="black")))
fig1.update_layout(plot_bgcolor="white", xaxis=dict(showgrid=True, gridcolor="lightgray"),
                   yaxis=dict(showgrid=True, gridcolor="lightgray"), legend_title_text="Stress Level")

fig2 = px.histogram(df, x='Age', color='Stress Category', barmode='overlay', nbins=30, opacity=0.6,
                    color_discrete_map=stress_colors, title="Age Distribution by Stress Level")
fig2.update_traces(marker=dict(line=dict(width=1, color="black")))
for cat, color in stress_colors.items():
    subset = df[df['Stress Category'] == cat]
    fig2.add_trace(go.Histogram(x=subset['Age'], histnorm='probability density', marker_color=color,
                               opacity=0.4, name=f"{cat} (Density)"))
fig2.update_layout(plot_bgcolor="white", xaxis=dict(showgrid=True, gridcolor="lightgray"),
                   yaxis=dict(showgrid=True, gridcolor="lightgray"), legend_title_text="Stress Level")

fig1.show()
fig2.show()


In [13]:
fig1 = px.violin(
    df, x="Physical Activity Level", y="Daily Steps", color="Physical Activity Level",
    box=True, points="all", hover_data=["Daily Steps", "Physical Activity Level"],
    title="Relationship Between Physical Activity Level and Daily Step Count",
    color_discrete_sequence=px.colors.qualitative.Pastel
).update_layout(
    xaxis_title="Physical Activity Level", yaxis_title="Daily Steps",
    plot_bgcolor="white", title_font=dict(size=18, color="black"), font=dict(size=14)
)

fig2 = px.density_heatmap(
    df, x="Sleep Duration", y="Heart Rate", z="Heart Rate", histfunc="avg",
    color_continuous_scale="Blues", title="Relationship Between Sleep Duration and Heart Rate"
).update_layout(
    xaxis_title="Sleep Duration (hours)", yaxis_title="Heart Rate (BPM)",
    plot_bgcolor="white", title_font=dict(size=18, color="black"), font=dict(size=14)
)

fig3 = px.histogram(
    df, x="Stress Level", color="Quality of Sleep", barmode="group", text_auto=True,
    color_discrete_sequence=px.colors.sequential.Purples,
    title="Relationship Between Stress Level and Sleep Quality"
).update_layout(
    xaxis_title="Stress Level", yaxis_title="Count",
    plot_bgcolor="white", title_font=dict(size=18, color="black"), font=dict(size=14)
)

fig1.show()
fig2.show()
fig3.show()


In [14]:
sleep_avg = df.groupby("Occupation")["Sleep Duration"].mean().reset_index()

fig1 = px.bar(
    sleep_avg, x="Occupation", y="Sleep Duration", color="Occupation",
    title="Average Sleep Duration by Occupation",
    labels={"Sleep Duration": "Average Sleep Duration (hours)"},
)
fig1.update_layout(xaxis_tickangle=-45, showlegend=False)

fig2 = px.box(
    df, x="Occupation", y="Stress Level", color="Occupation",
    title="Stress Level by Occupation",
    labels={"Stress Level": "Stress Level (scale 1-10)"},
)
fig2.update_layout(xaxis_tickangle=-45, showlegend=False)

top_occ = df["Occupation"].value_counts().nlargest(7).reset_index()
top_occ.columns = ["Occupation", "Count"]

fig3 = px.pie(
    top_occ, names="Occupation", values="Count",
    title="Top 7 Most Common Occupations", hole=0.4
)

fig1.show()
fig2.show()
fig3.show()

In [ ]:
import plotly.express as px

# Categorize heart rate
df["Heart Rate Category"] = df["Heart Rate"].apply(lambda x: "Elevated (>85 BPM)" if x > 85 else "Normal (≤85 BPM)")

# Count and percentage calculation
hr_counts = df["Heart Rate Category"].value_counts().reset_index()
hr_counts.columns = ["Heart Rate Category", "Count"]
hr_counts["Percentage"] = (hr_counts["Count"] / hr_counts["Count"].sum() * 100).round(1)

# Colors mapping
colors = {"Elevated (>85 BPM)": "#ff4c4c", "Normal (≤85 BPM)": "#34c759"}

# Pie chart
fig1 = px.pie(
    hr_counts, names="Heart Rate Category", values="Count",
    title="Heart Rate Analysis: Elevated vs Normal Levels",
    hole=0.4, color="Heart Rate Category",
    color_discrete_map=colors,
    hover_data=["Percentage"]
)
fig1.update_traces(textinfo="percent+label")
fig1.show()

# Bar chart
fig2 = px.bar(
    hr_counts, x="Heart Rate Category", y="Count", text="Count",
    title="Heart Rate Distribution: Elevated vs Normal",
    color="Heart Rate Category", color_discrete_map=colors,
    labels={"Count": "Number of People", "Heart Rate Category": "Category"}
)
fig2.update_layout(xaxis_tickangle=-15, yaxis_title="Number of People", showlegend=False)
fig2.show()


In [ ]:
# Define Elevated vs Normal Heart Rate (Threshold: 85 BPM)
df["Heart Rate Category"] = df["Heart Rate"].apply(lambda x: "Elevated (>85 BPM)" if x > 85 else "Normal (≤85 BPM)")

# Count occurrences
heart_rate_counts = df["Heart Rate Category"].value_counts().reset_index()
heart_rate_counts.columns = ["Heart Rate Category", "Count"]

# Calculate percentages
total = heart_rate_counts["Count"].sum()
heart_rate_counts["Percentage"] = (heart_rate_counts["Count"] / total * 100).round(1)

# PLOT 1: Pie Chart for Heart Rate Categories
fig1 = px.pie(heart_rate_counts, names="Heart Rate Category", values="Count",
              title="Heart Rate Analysis: Elevated vs Normal Levels",
              hole=0.4,
              color="Heart Rate Category",
              color_discrete_map={"Elevated (>85 BPM)": "#ff4c4c", "Normal (≤85 BPM)": "#34c759"},
              hover_data=["Percentage"])

fig1.update_traces(textinfo="percent+label")
fig1.show()

# PLOT 2: Bar Chart for Heart Rate Categories
fig2 = px.bar(heart_rate_counts, x="Heart Rate Category", y="Count", 
              title="Heart Rate Distribution: Elevated vs Normal",
              color="Heart Rate Category",
              text="Count",
              labels={"Count": "Number of People", "Heart Rate Category": "Category"},
              color_discrete_map={"Elevated (>85 BPM)": "#ff4c4c", "Normal (≤85 BPM)": "#34c759"})

fig2.update_layout(xaxis_tickangle=-15, yaxis_title="Number of People", showlegend=False)
fig2.show()

In [ ]:
# Define Sleep Duration Categories
df["Sleep Category"] = df["Sleep Duration"].apply(lambda x: "Low Sleep (<7 hours)" if x < 7 else "Normal Sleep (≥7 hours)")

# Count occurrences
sleep_counts = df["Sleep Category"].value_counts().reset_index()
sleep_counts.columns = ["Sleep Category", "Count"]

# Calculate percentages
total_sleep = sleep_counts["Count"].sum()
sleep_counts["Percentage"] = (sleep_counts["Count"] / total_sleep * 100).round(1)

# PLOT 1: Pie Chart for Sleep Duration Categories
fig1 = px.pie(sleep_counts, names="Sleep Category", values="Count",
              title="Sleep Duration Analysis: Less than 7 hours vs 7 hours and above",
              hole=0.4,  
              color="Sleep Category",
              color_discrete_map={"Low Sleep (<7 hours)": "#ff4c4c", "Normal Sleep (≥7 hours)": "#34c759"},
              hover_data=["Percentage"])

fig1.update_traces(textinfo="percent+label")
fig1.show()

# PLOT 2: Bar Chart for Sleep Duration Categories
fig2 = px.bar(sleep_counts, x="Sleep Category", y="Count", 
              title="Sleep Duration Distribution: Low vs Normal",
              color="Sleep Category",
              text="Count",
              labels={"Count": "Number of People", "Sleep Category": "Category"},
              color_discrete_map={"Low Sleep (<7 hours)": "#ff4c4c", "Normal Sleep (≥7 hours)": "#34c759"})

fig2.update_layout(xaxis_tickangle=-15, yaxis_title="Number of People", showlegend=False)
fig2.show()

In [ ]:
# Extract Systolic and Diastolic values
df[['Systolic', 'Diastolic']] = df['Blood Pressure'].str.split('/', expand=True).astype(int)

# Function to categorize blood pressure
def categorize_blood_pressure(systolic, diastolic):
    if systolic < 90 and diastolic < 60:
        return 'Low'
    elif 90 <= systolic < 120 and diastolic < 80:
        return 'Normal'
    elif 120 <= systolic < 130 and diastolic < 80:
        return 'Elevated'
    elif 130 <= systolic < 140 or 80 <= diastolic < 90:
        return 'Hypertension Stage 1'
    else:
        return 'Hypertension Stage 2'

# Applying categorization
df['Blood Pressure Category'] = df.apply(lambda row: categorize_blood_pressure(row['Systolic'], row['Diastolic']), axis=1)

# Scatter Density Plot
fig1 = px.scatter(df, x="Systolic", y="Diastolic", 
                  color="Blood Pressure Category",
                  title="Systolic vs. Diastolic Blood Pressure with Category Highlighting",
                  labels={"Systolic": "Systolic Blood Pressure", "Diastolic": "Diastolic Blood Pressure"},
                  category_orders={"Blood Pressure Category": ["Low", "Normal", "Elevated", "Hypertension Stage 1", "Hypertension Stage 2"]},
                  color_discrete_map={
                      "Low": "#1f77b4",
                      "Normal": "#2ca02c",
                      "Elevated": "#ff7f0e",
                      "Hypertension Stage 1": "#d62728",
                      "Hypertension Stage 2": "#9467bd"
                  })

fig1.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=1, color="black")))

fig1.update_layout(
    xaxis_title="Systolic Blood Pressure",
    yaxis_title="Diastolic Blood Pressure",
    template="plotly_dark",
    legend_title="Blood Pressure Category",
    hovermode="closest"
)

fig1.show()

In [ ]:
# Remove missing values and "None" cases (No Disorder)
df_filtered = df.dropna(subset=["Sleep Disorder"])  
df_filtered = df_filtered[df_filtered["Sleep Disorder"] != "None"]

# Bar Chart: Sleep Disorder by Gender
fig1 = px.bar(df_filtered.groupby(["Gender", "Sleep Disorder"]).size().reset_index(name="Count"), 
              x="Gender", y="Count", color="Sleep Disorder",
              title="🛌 Sleep Disorders by Gender (Excluding No Disorder)",
              labels={"Count": "Number of Cases", "Gender": "Gender"},
              barmode="group",
              color_discrete_sequence=px.colors.qualitative.Dark24,  
              text_auto=True)  

fig1.update_layout(
    xaxis_title="🧑‍🤝‍🧑 Gender",
    yaxis_title="📊 Number of Cases",
    template="plotly_dark",
    font=dict(size=14),
    hovermode="x unified",
    legend=dict(title="🛌 Sleep Disorder Type", orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
)

# Pie Chart: Gender Proportion in Sleep Disorders
fig2 = px.pie(df_filtered, names="Gender", 
              title="🔵 Gender Proportion in Sleep Disorder Cases (Excluding No Disorder)",
              color_discrete_sequence=px.colors.qualitative.Pastel,
              hole=0.4)

fig2.update_traces(textinfo='percent+label', pull=[0.1, 0])  

# Heatmap: Sleep Disorder Prevalence by Gender
heatmap_data = df_filtered.groupby(["Gender", "Sleep Disorder"]).size().reset_index(name="Count")

fig3 = px.density_heatmap(heatmap_data, 
                          x="Sleep Disorder", 
                          y="Gender", 
                          z="Count", 
                          text_auto=True,
                          title="🌡️ Heatmap of Sleep Disorder Prevalence by Gender",
                          color_continuous_scale="Viridis")

fig3.update_layout(
    xaxis_title="🛌 Sleep Disorder Type",
    yaxis_title="🧑‍🤝‍🧑 Gender",
    template="plotly_white"
)

# Sunburst Chart: Sleep Disorders Breakdown
df_sunburst = df_filtered.groupby(["Gender", "Sleep Disorder"]).size().reset_index(name="Count")

fig4 = px.sunburst(df_sunburst, 
                   path=["Gender", "Sleep Disorder"],  
                   values="Count",  # Aggregated counts
                   title="🌞 Sleep Disorders Breakdown by Gender",
                   color="Sleep Disorder",
                   color_discrete_sequence=px.colors.qualitative.Prism)

fig1.show()
fig2.show()
fig3.show()
fig4.show()

In [ ]:
# Convert categorical columns into numerical (One-Hot Encoding or Label Encoding)
df_encoded = df.copy()

# Identify categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns

# Encode categorical variables using label encoding
for col in categorical_columns:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

# Compute the correlation matrix
corr_matrix = df_encoded.corr()

# Heatmap
fig = ff.create_annotated_heatmap(
    z=corr_matrix.values, 
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale="Viridis",
    annotation_text=corr_matrix.round(2).values,
    hoverinfo="z"
)

fig.update_layout(
    title="",
    xaxis_title="Features",
    yaxis_title="Features",
    template="plotly_dark",
    width=1000, height=800
)

fig.show()

In [ ]:
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 67.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 MB 63.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import Libraries
import shap
import xgboost as xgb
import lightgbm as lgb
import joblib  # To save the best model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings


# Suppress warnings
warnings.filterwarnings("ignore")


# Standardizing 'BMI Category'
df["BMI Category"] = df["BMI Category"].replace("Normal Weight", "Normal")

# Ensure 'Blood Pressure' is a string before splitting
df["Blood Pressure"] = df["Blood Pressure"].astype(str)

# Splitting 'Blood Pressure' into Systolic & Diastolic columns
df[["Systolic Pressure", "Diastolic Pressure"]] = df["Blood Pressure"].str.split("/", expand=True)

# Convert to numeric
df["Systolic Pressure"] = pd.to_numeric(df["Systolic Pressure"], errors="coerce")
df["Diastolic Pressure"] = pd.to_numeric(df["Diastolic Pressure"], errors="coerce")

# Drop original 'Blood Pressure' column
df.drop(columns=["Blood Pressure"], inplace=True)

# Handle missing values
df.dropna(subset=["Systolic Pressure", "Diastolic Pressure"], inplace=True)

# Encode Categorical Variables
df_encoded = df.copy()
label_encoders = {}

for col in ["BMI Category", "Sleep Disorder", "Gender", "Occupation"]:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# Define Features (X) & Target Variable (y)
X = df_encoded.drop(columns=["Person ID", "Sleep Disorder"])
y = df_encoded["Sleep Disorder"]

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Feature Selection using SHAP
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# SHAP Summary Plot (COMMENTED OUT)
# shap.summary_plot(shap_values, X_test, feature_names=X.columns)

# Feature Selection: Keep only important features
feature_importance = pd.DataFrame(
    {"Feature": X.columns, "Importance": rf_model.feature_importances_}
)
selected_features = feature_importance[feature_importance["Importance"] > 0.05]["Feature"].tolist()

# Apply selected features
X_train = X_train[selected_features]
X_test = X_test[selected_features]

# Hyperparameter Tuning for Random Forest
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42), param_grid, cv=3, scoring="accuracy"
)
grid_search.fit(X_train, y_train)

# Train Best Model
best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

# Classification Report - Random Forest
report_rf = classification_report(y_test, y_pred_rf, output_dict=True)
report_rf_df = pd.DataFrame(report_rf).transpose().reset_index()
fig_rf_table = ff.create_table(report_rf_df.round(2))
fig_rf_table.update_layout(title_text="Classification Report - Random Forest")
fig_rf_table.show()

# Confusion Matrix - Random Forest
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
fig_rf_heatmap = go.Figure(
    data=go.Heatmap(
        z=conf_matrix_rf,
        x=y.unique(),
        y=y.unique(),
        colorscale="Blues",
        text=conf_matrix_rf,
        texttemplate="%{text}"
    )
)
fig_rf_heatmap.update_layout(title="Confusion Matrix - Random Forest", xaxis_title="Predicted", yaxis_title="Actual")
fig_rf_heatmap.show()

# Train XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=10, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# Classification Report - XGBoost
report_xgb = classification_report(y_test, y_pred_xgb, output_dict=True)
report_xgb_df = pd.DataFrame(report_xgb).transpose().reset_index()
fig_xgb_table = ff.create_table(report_xgb_df.round(2))
fig_xgb_table.update_layout(title_text="Classification Report - XGBoost")
fig_xgb_table.show()

# Confusion Matrix - XGBoost
conf_matrix_xgb = confusion_matrix(y_test, y_pred_xgb)
fig_xgb_heatmap = go.Figure(
    data=go.Heatmap(
        z=conf_matrix_xgb,
        x=y.unique(),
        y=y.unique(),
        colorscale="Oranges",
        text=conf_matrix_xgb,
        texttemplate="%{text}"
    )
)
fig_xgb_heatmap.update_layout(title="Confusion Matrix - XGBoost", xaxis_title="Predicted", yaxis_title="Actual")
fig_xgb_heatmap.show()

# Train LightGBM
lgb_model = lgb.LGBMClassifier(n_estimators=200, max_depth=10, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

# Classification Report - LightGBM
report_lgb = classification_report(y_test, y_pred_lgb, output_dict=True)
report_lgb_df = pd.DataFrame(report_lgb).transpose().reset_index()
fig_lgb_table = ff.create_table(report_lgb_df.round(2))
fig_lgb_table.update_layout(title_text="Classification Report - LightGBM")
fig_lgb_table.show()

# Confusion Matrix - LightGBM
conf_matrix_lgb = confusion_matrix(y_test, y_pred_lgb)
fig_lgb_heatmap = go.Figure(
    data=go.Heatmap(
        z=conf_matrix_lgb,
        x=y.unique(),
        y=y.unique(),
        colorscale="Greens",
        text=conf_matrix_lgb,
        texttemplate="%{text}"
    )
)
fig_lgb_heatmap.update_layout(title="Confusion Matrix - LightGBM", xaxis_title="Predicted", yaxis_title="Actual")
fig_lgb_heatmap.show()

# Compare Model Performance
models_df = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost", "LightGBM"],
    "Accuracy": [accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_xgb), accuracy_score(y_test, y_pred_lgb)]
}).sort_values(by="Accuracy", ascending=False)

# Plot Comparison Table
fig_models = go.Figure(
    data=[go.Bar(
        x=models_df["Model"],
        y=models_df["Accuracy"],
        text=models_df["Accuracy"].round(2),
        textposition="auto",
        marker=dict(color=["blue", "orange", "green"])
    )]
)
fig_models.update_layout(title="Model Accuracy Comparison", xaxis_title="Model", yaxis_title="Accuracy", yaxis=dict(range=[0, 1]))
fig_models.show()

import joblib  # which helps to save models

# Save Best Model
best_model_name = models_df.iloc[0]["Model"]
if best_model_name == "Random Forest":
    joblib.dump(best_rf, "best_model.pkl")
    print("Best model saved: Random Forest")
elif best_model_name == "XGBoost":
    joblib.dump(xgb_model, "best_model.pkl")
    print("Best model saved: XGBoost")
else:
    joblib.dump(lgb_model, "best_model.pkl")
    print("Best model saved: LightGBM")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 99
[LightGBM] [Info] Number of data points in the train set: 299, number of used features: 7
[LightGBM] [Info] Start training from score -1.573309
[LightGBM] [Info] Start training from score -1.573309
[LightGBM] [Info] Start training from score -0.535658
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

Best model saved: XGBoost
